# Evaluation & fair comparison (run after the three training notebooks)

Produces everything for the thesis results chapter and saves it to `MyDrive/RDD_fair/results/`:
per-run metrics in each model's folder, seed summaries in `<model>/summary/`, and cross-model
tables, efficiency table, D20/D40 error analysis, GT-vs-predictions grids (test set AND Albania),
speed/efficiency benchmark and the reproducibility appendix in `results/comparison/`.

Outputs covered end to end: **Precision, Recall, F1, mAP50, mAP50-95 (overall + per class),
Params (M), FP32 weights (MB), GFLOPs, inference time (ms/img), FPS, and Ground-Truth-vs-predictions
image grids** for both the RDD2022 test set and the Albania held-out set.

Use the **same GPU type** as for training (the speed/efficiency benchmark depends on it).

In [ ]:
# 1) Mount Drive and import the shared code (upload the folder rdd_fair/ to MyDrive/RDD_fair/code/)
from google.colab import drive
drive.mount('/content/drive')
import sys, os
CODE_DIR = '/content/drive/MyDrive/RDD_fair/code'
sys.path.insert(0, CODE_DIR)
!nvidia-smi

In [ ]:
# 2) Same library versions in every session, same GPU type, same dataset
from rdd_fair import config as C, utils as U, prepare_data as P
U.pin_or_install_packages()
C.ensure_dirs()
U.check_gpu_consistency()
data_yaml = P.prepare_rdd()
albania_json = P.prepare_albania()   # held-out, used only here
!pip install -q tabulate thop

In [ ]:
# 3) Predictions with each selected checkpoint (val -> threshold, test, Albania); cached on Drive
from rdd_fair import report as R
for m in C.MODEL_CFG:
    for s in C.SEEDS:
        R.run_predictions(m, s, splits=('val', 'test', 'albania'))

In [ ]:
# 4) Per-run evaluation: COCO mAP, per-class AP/P/R/F1, bootstrap 95% CI, confusion matrix, error analysis
for m in C.MODEL_CFG:
    for s in C.SEEDS:
        if os.path.exists(os.path.join(C.run_dir(m, s, 'predictions'), 'test.json')):
            R.evaluate_run(m, s, eval_splits=('test', 'albania'))
            print('evaluated', m, s)

In [ ]:
# 5) Mean +- std over seeds, test set and Albania: Precision, Recall, F1, mAP50, mAP50-95 (overall + per class)
from IPython.display import Markdown
for split in ('test', 'albania'):
    overall, tabs = R.compare_models(split)
    if overall.empty:
        continue
    display(Markdown(f'## {split}: overall (mean +/- std over seeds)')); display(overall)
    for k in ('AP50', 'AP50_95', 'precision', 'recall', 'F1'):
        display(Markdown(f'**Per-class {k}**')); display(tabs[k])

In [ ]:
# 6) Confidence intervals of seed-averaged metrics + paired model differences (same resampled images)
for split in ('test', 'albania'):
    try:
        ci, diff = R.seed_averaged_paired_bootstrap(split)
    except Exception as e:
        print(f'skipping bootstrap for {split}: {e}')
        continue
    display(Markdown(f'## {split}: 95% bootstrap CI')); display(ci[ci.metric.isin(['mAP50', 'mAP50_95', 'macro_F1'])])
    display(Markdown(f'## {split}: paired differences (significant if CI excludes 0)'))
    display(diff[diff.metric.isin(['mAP50', 'mAP50_95', 'macro_F1', 'D20_AP50', 'D40_AP50'])])

In [ ]:
# 7) Error analysis for D20 / D40 (the four classes are D00, D10, D20, D40; see the paper, Sect. 3.2)
from IPython.display import Image as IImage
import pandas as pd
for split in ('test', 'albania'):
    R.focus_error_analysis(split)
    p = os.path.join(C.COMPARISON_DIR, f'{split}_focus_errors_mean_over_seeds.csv')
    if not os.path.exists(p):
        continue
    display(Markdown(f'## {split}: D20/D40 error composition')); display(pd.read_csv(p))
    for f in (f'{split}_focus_error_composition.png', f'{split}_focus_ap_vs_iou.png', f'{split}_focus_recall_by_size.png'):
        fp = os.path.join(C.COMPARISON_DIR, f)
        if os.path.exists(fp):
            display(IImage(fp))
    for m in C.MODEL_CFG:
        if os.path.exists(os.path.join(C.run_dir(m, 0, 'metrics'), f'{split}_fp_records.csv')):
            R.error_gallery(m, seed=0, split=split)
            gp = os.path.join(C.run_dir(m, 0, 'figures'), 'error_galleries', f'{split}_D40_gallery.png')
            if os.path.exists(gp):
                display(IImage(gp))

In [ ]:
# 8) Efficiency table: Params (M), FP32 weights (MB), GFLOPs, inference time (ms/img), FPS
#    (batch 1, FP32, same GPU/session for all three models -- single source of truth, replaces
#    any ad-hoc efficiency cell)
eff_df = R.efficiency_table(seed=0)
display(Markdown('## Efficiency (batch 1, FP32)')); display(eff_df)

In [ ]:
# 9) Master summary tables: Precision/Recall/F1/mAP50/mAP50-95 + Params(M)/weights/GFLOPs/Time/FPS,
#    overall and per-class, for both the RDD2022 test set and the Albania held-out set.
for split in ('test', 'albania'):
    overall_df, perclass_df = R.master_table(split, efficiency=eff_df)
    if overall_df.empty:
        continue
    display(Markdown(f'## {split}: master summary table')); display(overall_df)
    display(Markdown(f'## {split}: master per-class table')); display(perclass_df)

In [ ]:
# 10) Ground truth vs. predictions, side by side (GT | SSD | Faster R-CNN | YOLO), for BOTH splits.
#     Albania (200 images total) also gets every single image saved, annotated per model.
for split in ('test', 'albania'):
    paths = R.prediction_grid(split=split, seed=0, n_images=12)
    display(Markdown(f'## {split}: ground truth vs. predictions'))
    for p in paths:
        display(IImage(p))

for m in C.MODEL_CFG:
    if os.path.exists(os.path.join(C.run_dir(m, 0, 'predictions'), 'albania.json')):
        R.save_all_albania_predictions(m, seed=0)
print('Saved every annotated Albania image per model under results/<model>/seed0/figures/albania_predictions/')

In [ ]:
# 11) Training curves, and the reproducibility appendix (data/split, protocol, per-model settings,
#     hardware/software per run, documented architectural differences)
R.training_curves()
display(IImage(os.path.join(C.COMPARISON_DIR, 'training_curves_val.png')))
display(Markdown(open(R.reproducibility_appendix()).read()))

In [ ]:
from IPython.display import Image as IImage
import cv2, matplotlib.pyplot as plt, os

MY_STEMS = [
    'Albania_084','Albania_081','Albania_003','Albania_004','Albania_005','Albania_006',
    'Albania_010','Albania_009','Albania_015','Albania_023','Albania_022','Albania_024',
    'Albania_041','Albania_048','Albania_051','Albania_064','Albania_145','Albania_095',
    'Albania_067','Albania_066','Albania_099','Albania_103','Albania_112','Albania_135',
    'Albania_155','Albania_169','Albania_167','Albania_189',
]

seed = 0
MODEL_ORDER = ['yolov8m', 'ssd300_vgg16', 'fasterrcnn_r50_fpn_v2']  # GT | YOLO | SSD | Faster R-CNN
models = [m for m in MODEL_ORDER if os.path.exists(os.path.join(C.run_dir(m, seed, 'predictions'), 'albania.json'))]
dets = {m: R.load_dets(m, seed, 'albania') for m in models}
thr = {m: R.load_json(os.path.join(C.run_dir(m, seed, 'metrics'), 'operating_threshold.json'))['threshold'] for m in models}

ev = R.evaluator('albania')

# match by PREFIX instead of exact stem (Roboflow appends _jpg.rf.<hash>.jpg etc.)
def find_match(stem):
    hits = [iid for iid, im in ev.images.items() if im['file_name'].startswith(stem)]
    return hits

out_dir = os.path.join(C.COMPARISON_DIR, 'albania_individual_grids')
os.makedirs(out_dir, exist_ok=True)

for stem in MY_STEMS:
    hits = find_match(stem)
    if not hits:
        print('NOT FOUND:', stem)
        continue
    if len(hits) > 1:
        print(f'WARNING: {stem} matched {len(hits)} files, using the first:',
              [ev.images[h]['file_name'] for h in hits])
    iid = hits[0]
    im = ev.images[iid]
    img = cv2.cvtColor(cv2.imread(os.path.join(P.img_dir('albania'), im['file_name'])), cv2.COLOR_BGR2RGB)
    panels = [('Ground truth', R._draw(img, R._gt_boxes(ev, iid)))]
    for m in models:
        panels.append((f'{C.MODEL_LABELS[m]} (conf>={thr[m]:.2f})', R._draw(img, R._pred_boxes(dets[m], iid, thr[m]))))

    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.2))
    for c, (t, p) in enumerate(panels):
        axes[c].imshow(p); axes[c].set_axis_off(); axes[c].set_title(t, fontsize=10)
    fig.suptitle(stem, fontsize=12)
    fig.tight_layout()
    out_path = os.path.join(out_dir, f'{stem}_grid.png')
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print('saved', out_path)

print('done')

In [ ]:
from IPython.display import Image as IImage
import cv2, collections, numpy as np, matplotlib.pyplot as plt, os

split = 'test'
seed = 0
N_PER_CLASS = 2
pick_seed = 0   # change this to get a different random sample

MODEL_ORDER = ['yolov8m', 'ssd300_vgg16', 'fasterrcnn_r50_fpn_v2']  # GT | YOLO | SSD | Faster R-CNN
models = [m for m in MODEL_ORDER if os.path.exists(os.path.join(C.run_dir(m, seed, 'predictions'), f'{split}.json'))]
dets = {m: R.load_dets(m, seed, split) for m in models}
thr = {m: R.load_json(os.path.join(C.run_dir(m, seed, 'metrics'), 'operating_threshold.json'))['threshold'] for m in models}

ev = R.evaluator(split)
rng = np.random.default_rng(pick_seed)

# group image ids by class present in their ground truth
by_class = collections.defaultdict(list)
for iid in ev.img_ids:
    for cid in ev.cat_ids:
        if len(ev.gt_boxes[(iid, cid)]):
            by_class[ev.cat_names[cid]].append(iid)

chosen = {}  # class -> list of chosen image ids
used = set()
for c in C.CLASSES:
    cand = [i for i in by_class.get(c, []) if i not in used]
    picks = list(rng.choice(cand, min(N_PER_CLASS, len(cand)), replace=False)) if cand else []
    chosen[c] = picks
    used.update(picks)

out_dir = os.path.join(C.COMPARISON_DIR, f'{split}_individual_grids')
os.makedirs(out_dir, exist_ok=True)

for c, iids in chosen.items():
    for k, iid in enumerate(iids):
        im = ev.images[int(iid)]
        img = cv2.cvtColor(cv2.imread(os.path.join(P.img_dir(split), im['file_name'])), cv2.COLOR_BGR2RGB)
        panels = [('Ground truth', R._draw(img, R._gt_boxes(ev, int(iid))))]
        for m in models:
            panels.append((f'{C.MODEL_LABELS[m]} (conf>={thr[m]:.2f})', R._draw(img, R._pred_boxes(dets[m], int(iid), thr[m]))))

        fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.2))
        for p_i, (t, p) in enumerate(panels):
            axes[p_i].imshow(p); axes[p_i].set_axis_off(); axes[p_i].set_title(t, fontsize=10)
        fig.suptitle(f'{c} — {im["file_name"]}', fontsize=11)
        fig.tight_layout()
        out_path = os.path.join(out_dir, f'{c}_{k+1}_{os.path.splitext(im["file_name"])[0]}.png')
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print('saved', out_path)

print('done —', sum(len(v) for v in chosen.values()), 'grids saved to', out_dir)

In [ ]:
# 11) Peak GPU memory, inference (batch 1, FP32, 550 RDD2022 test images)
import torch, glob
from PIL import Image
import torchvision.transforms.functional as TF
from torchvision.models.detection import ssd300_vgg16, fasterrcnn_resnet50_fpn_v2
from ultralytics import YOLO

paths = sorted(glob.glob(os.path.join(P.img_dir('test'), '*')))[:550]
imgs = [TF.to_tensor(Image.open(p).convert('RGB')) for p in paths]

def load_sd(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(ck, dict):
        for k in ['ema', 'model_ema', 'ema_state_dict', 'model', 'model_state_dict', 'state_dict']:
            if k in ck and ck[k] is not None:
                ck = ck[k]; break
    if hasattr(ck, 'state_dict'): ck = ck.state_dict()
    return {(k[7:] if k.startswith('module.') else k): v for k, v in ck.items() if k != 'n_averaged'}

def peak(run):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    run(); torch.cuda.synchronize()
    return round(torch.cuda.max_memory_allocated() / 1024**2)

def run_tv(model):
    model.eval().cuda()
    with torch.no_grad():
        for im in imgs:
            model([im.cuda()])

res = {}
sd = load_sd(os.path.join(C.run_dir('ssd300_vgg16', 0, 'checkpoints'), 'best.pt'))
m = ssd300_vgg16(weights=None, weights_backbone=None,
                 num_classes=sd['head.classification_head.module_list.0.weight'].shape[0] // 4)
print(m.load_state_dict(sd, strict=False)); res['SSD300-VGG16'] = peak(lambda: run_tv(m)); del m

sd = load_sd(os.path.join(C.run_dir('fasterrcnn_r50_fpn_v2', 0, 'checkpoints'), 'best.pt'))
m = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, min_size=640, max_size=640,
                               num_classes=sd['roi_heads.box_predictor.cls_score.weight'].shape[0])
print(m.load_state_dict(sd, strict=False)); res['Faster R-CNN'] = peak(lambda: run_tv(m)); del m

y = YOLO(os.path.join(C.run_dir('yolov8m', 0, 'checkpoints'), 'best.pt'))
res['YOLOv8m'] = peak(lambda: [y.predict(p, imgsz=640, device=0, conf=0.001, iou=0.5,
                                         max_det=100, verbose=False) for p in paths])
print('Peak GPU memory, inference (MB):', res)